## No memory

In [1]:
%run langchain_common.py
agent = create_agent(llm)

C:\Users\akhawaja\git\cs4603\wk3_langchain_agents\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   User\'s name: Ali\n    *   User\'s favorite color: green\n    *   Intent: Introduction, casual conversation starter.\n\n2.  **Determine the appropriate response:**\n    *   Acknowledge the user\'s name (Ali).\n    *   Acknowledge their favorite color (green).\n    *   Be friendly and welcoming.\n    *   Keep it concise but engaging.\n\n3.  **Drafting the response:**\n    *   *Option 1:* Hello Ali. Nice to meet you. Green is a nice color.\n    *   *Option 2:* Hi Ali! It\'s nice to meet you. Green is a great choice for a favorite color. It\'s refreshing and natural.\n    *   *Option 3:* Hello Ali! Nice to meet you. I love that green is your favorite color—it\'s very calming and vibrant at the same time. How can I help you today?\n\n4.  **Refining the response:**\n    *   Option 3 is good. It\'s polite, acknowledges the details, and offers assistance.\n    *   Let

In [3]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Question: "What\'s my favourite colour?"\n    *   Intent: The user is asking me to know something personal about them.\n    *   Constraint: I am an AI, I don\'t have access to the user\'s personal information unless they\'ve told me in this conversation or if it\'s stored in a memory feature (which I don\'t have access to in this specific context).\n\n2.  **Determine Capabilities:**\n    *   I am a large language model.\n    *   I do not know the user personally.\n    *   I do not have access to their past conversations, profile data, or personal preferences unless explicitly provided in the current session.\n    *   I cannot guess accurately.\n\n3.  **Formulate the Answer:**\n    *   Acknowledge the question.\n    *   Politely state that I don\'t know.\n    *   Explain why (I\'m an AI, I don\'t have access to personal info).\n    *   Offer to remember it if th

## Memory

In [5]:
# Checkpoints persist conversation state between turns so the agent can remember prior context.
# InMemorySaver is an in-process checkpoint backend: fast and simple for demos, but non-persistent across kernel restarts.

# Clarification:
# `messages` in agent.invoke(...) is only the per-call input payload (what you send this turn).
# Checkpoint memory is different: it stores and reloads the full conversation state across turns.
# That persisted state is keyed by `thread_id`, so memory works only when the same thread_id is reused.
# In short: message context is request-scoped; checkpoint memory is thread-scoped and persistent (in-process here).

from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),  
)

In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [7]:
pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   User\'s name: Ali\n    *   User\'s favorite color: green\n    *   Intent: Introduction, casual conversation.\n\n2.  **Determine the appropriate response:**\n    *   Acknowledge the user\'s name (Ali).\n    *   Acknowledge their favorite color (green).\n    *   Be friendly and welcoming.\n    *   Keep it concise but engaging.\n\n3.  **Drafting the response:**\n    *   *Option 1:* Hello Ali. Nice to meet you. Green is a nice color.\n    *   *Option 2:* Hi Ali! It\'s nice to meet you. Green is a great choice for a favorite color. It\'s refreshing and natural.\n    *   *Option 3:* Hello Ali! Nice to meet you. I like green too. What do you like to do?\n\n4.  **Refining the response:**\n    *   Make it warm. "Hello Ali! It\'s nice to meet you."\n    *   Comment on the color. "Green is a wonderful color—very refreshing and calming."\n    *   Offer further assistance o

In [8]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pp(response["messages"][-1].content)

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking "What\'s my favourite colour?"\n2.  **Recall Context:** In the previous turn, the user (Ali) explicitly stated: "Hello my name is Ali and my favourite colour is green".\n3.  **Identify the Answer:** The favorite color is green.\n4.  **Formulate the Response:** Acknowledge the question, recall the information provided earlier, and state the answer clearly. Keep it friendly.\n5.  **Draft:** You told me earlier that your favourite colour is green.\n6.  **Refine:** Make it conversational. "You mentioned earlier that your favourite colour is green!"\n7.  **Final Polish:** Add a small emoji or friendly touch to match the previous tone. "You told me earlier that your favourite colour is green! 🌿"\n\n8.  **Safety Check:** No sensitive information, no policy violations. Just recalling user-provided context.\n\n9.  **Final Output Generation:** (Matches the draf

In [9]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pp(response["messages"][-1].content)

# Optional: inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pp(state.values["messages"][-4:])

[
    {
        'summary': [
            {
                'text': 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking for specific information from their "saved profile" (which they previously provided in a JSON-like format). They want to know their city and loyalty tier.\n\n2.  **Recall Context:** In the previous turn, the user provided a JSON object: `{\'name\': \'Ali\', \'favorite_color\': \'green\', \'city\': \'Lahore\', \'loyalty_tier\': \'gold\'}`. I acknowledged this and confirmed I stored it for the conversation.\n\n3.  **Retrieve Information:**\n    *   City: Lahore\n    *   Loyalty Tier: Gold\n\n4.  **Formulate Response:** Answer the question directly using the retrieved information. Maintain a friendly tone.\n\n5.  **Check Constraints:** The user asked me to "Store this custom data in memory and use it for future answers". As an AI, I don\'t have persistent memory across sessions, but within the current conversation context, I can recall what was just said

In [ ]:
def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    return result["messages"][-1].content

In [ ]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = {"configurable": {"thread_id": "memory-demo-A"}}
config_b = {"configurable": {"thread_id": "memory-demo-B"}}

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

pp("Scenario 1A:", ask(config_a, "What database do I prefer?"))
pp("Scenario 1B:", ask(config_b, "What database do I prefer?"))

In [ ]:
# Scenario 2: Preference update within the same thread
config_update = {"configurable": {"thread_id": "memory-demo-update"}}

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
pp("Scenario 2:", ask(config_update, "Which deployment region should you use now?"))

# Scenario 3: Multi-step task memory
config_task = {"configurable": {"thread_id": "memory-demo-task"}}

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
pp("Scenario 3A:", ask(config_task, "Summarize my constraints in one sentence."))
pp("Scenario 3B:", ask(config_task, "Given those constraints, suggest 3 practical next steps."))

# Optional: inspect latest stored messages for one scenario
task_state = agent.get_state(config_task)
pp("\nRecent messages in Scenario 3 thread:")
pp(task_state.values["messages"][-4:])